## Paso: Leer el entrypoint de simulación
Queremos saber qué formato de archivo espera `create_hpo_dataset.py` como input.
Las primeras 60 líneas siempre incluyen los imports, el argparse y la apertura del archivo de datos.

In [1]:
%%bash
head -60 ~/lucai/GraPhens/src/simulation/phenotype_simulation/create_hpo_dataset.py

import os
import json
import torch
import numpy as np
import logging
import time  # Import time for logging
from pathlib import Path
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional
from torch_geometric.data import Data, Dataset
from torch.utils.data import random_split

# Import GraPhens for graph creation
from src.graphens import GraPhens

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class HPOGraphDataset(Dataset):
    """
    PyTorch Dataset for HPO graphs created from simulated patient data.
    
    This dataset:
    1. Loads a preprocessed index file
    2. On-demand loads individual graph objects
    3. Supports shuffling and random access
    """
    
    def __init__(self, root_dir: str, transform=None, pre_transform=None):
        """
        Initialize HPO Graph Dataset.
        
        Args:
            root_dir: Directory containing processed datase

## Buscar dónde lee el archivo de gen-fenotipo
Primero ubicamos las funciones definidas, después leemos la que nos interesa.

In [2]:
%%bash
grep -n "def " ~/lucai/GraPhens/src/simulation/phenotype_simulation/create_hpo_dataset.py

30:    def __init__(self, root_dir: str, transform=None, pre_transform=None):
58:    def len(self):
61:    def get(self, idx):
80:    def get_gene_mask(dataset, gene):
89:def convert_to_dataset(
278:def create_train_val_test_splits(


## Leer convert_to_dataset — el corazón del generador
Esta función toma el .txt de gen-fenotipo y produce el dataset. Necesitamos ver
qué argumentos recibe y cómo abre el archivo de datos.

In [3]:
%%bash
sed -n '89,140p' ~/lucai/GraPhens/src/simulation/phenotype_simulation/create_hpo_dataset.py

def convert_to_dataset(
    input_json: str,
    output_dir: str,
    batch_size: int = 1000,
    max_samples_per_gene: Optional[int] = None,
    embedding_lookup_path: str = "data/embeddings/hpo_embeddings_latest.pkl",
    include_ancestors: bool = True,
    include_reverse_edges: bool = True,
    clear_existing: bool = False
):
    """
    Convert simulated patient data from JSON to PyTorch Geometric dataset.
    
    Args:
        input_json: Path to input JSON file with simulated patient data
        output_dir: Directory to save processed dataset
        batch_size: Number of samples to process at once
        max_samples_per_gene: Maximum samples to use per gene (useful for testing)
        embedding_lookup_path: Path to pre-computed HPO embeddings
        include_ancestors: Whether to augment with ancestor HPO terms
        include_reverse_edges: Whether to include reverse edges in the graph
        clear_existing: Whether to clear existing processed files
    """
    total_star

Dato importante: el input no es el .txt directamente — es un input_json. El pipeline tiene un paso anterior que todavía no vimos.
El flujo real es:

phenotype_to_genes.txt

        ↓
   ¿algo que genera el JSON?    ← esto no lo vimos todavía
   
        ↓
   convert_to_dataset(input_json=...)
   
        ↓
   dataset .pt en output_dir

        ↓
   training.py
   


## Buscar el generador del JSON de simulación
El input de convert_to_dataset es un JSON. Buscamos qué script lo genera
y qué estructura tiene.

In [4]:
%%bash
# Buscar en todo el directorio simulation/ qué archivos mencionan phenotype_to_genes o el .txt
grep -rn "phenotype_to_genes\|\.txt" ~/lucai/GraPhens/src/simulation/ | grep -v "__pycache__"

grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/analysis.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/keep_leaves_of_goldstandard_dataset.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/update_goldstandard_dataset.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/update_deprecated_hpo.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/gene_phenotype/__pycache__/demo.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/gene_phenotype/__pycache__/database.cpython-313.pyc: coincidencia en fichero binario


/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:165:    phenotype_to_genes_rows = []
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:201:        # Add to phenotype_to_genes rows
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:202:        phenotype_to_genes_rows.append({
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:219:        phenotypes_df = pd.DataFrame(phenotype_to_genes_rows)
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:220:        database._phenotype_to_genes = pd.concat(
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:221:            [database._phenotype_to_genes, phenotypes_df], ignore_index=True
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:283:            os.path.join(str(data_dir), "genes_to_phenotype.txt"),
/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:284:            os.path.j

## Leer las columnas exactas que espera database.py
Necesitamos saber las columnas de phenotype_to_genes.txt para comparar
con las de nuestro ot_unified_genes_phenotypes.txt.

In [5]:
%%bash
sed -n '47,60p' ~/lucai/GraPhens/src/simulation/gene_phenotype/database.py

    def _load_phenotype_to_genes(self) -> None:
        """Load the phenotype_to_genes.txt file."""
        file_path = os.path.join(self.data_dir, "phenotype_to_genes.txt")
        self._phenotype_to_genes = pd.read_csv(
            file_path, 
            sep='\t',
            dtype={
                'hpo_id': str,
                'hpo_name': str,
                'ncbi_gene_id': str,
                'gene_symbol': str,
                'disease_id': str
            }
        )


## Ver qué hay en phenotype_simulation/ — buscar el generador del JSON

In [6]:
%%bash
ls ~/lucai/GraPhens/src/simulation/phenotype_simulation/

README.md
__init__.py
__pycache__
ablation_simulation.py
create_hpo_dataset.py
data_loader.py
demo.py
distributions.py
factory.py
integration_demo.py
interfaces.py
selector.py
simulator.py


## Comparar columnas del dataset OT contra lo que espera database.py

`database.py` carga `phenotype_to_genes.txt` con pd.read_csv(sep='\t') y espera exactamente
5 columnas: hpo_id, hpo_name, ncbi_gene_id, gene_symbol, disease_id.

Nuestro `ot_unified_genes_phenotypes.txt` fue construido a partir de 6 fuentes OT distintas,
cada una con su propio script. No está garantizado que tenga esas mismas columnas ni ese mismo
separador — puede que tenga columnas de más, de menos, o con nombres distintos.

Si las columnas coinciden exactamente, podemos apuntar database.py directamente al archivo OT.
Si no, necesitamos un paso de transformación antes de entrar al pipeline.

In [1]:
import pandas as pd
from pathlib import Path

ot = pd.read_csv(
    Path.home() / "lucai/datos/ot_unified_genes_phenotypes.txt",
    sep="\t",
    nrows=5
)

print("Columnas del dataset OT:")
print(ot.columns.tolist())
print()
print(ot.head())

Columnas del dataset OT:
['hpo_id', 'hpo_name', 'ncbi_gene_id', 'gene_symbol', 'disease_id', 'fuente']

       hpo_id                 hpo_name  ncbi_gene_id gene_symbol   disease_id  \
0  HP:0002188  Delayed CNS myelination        199857       ALG14  OMIM:619031   
1  HP:0002188  Delayed CNS myelination        199857       ALG14  OMIM:619036   
2  HP:0002188  Delayed CNS myelination          9131       AIFM1  OMIM:300232   
3  HP:0002188  Delayed CNS myelination         26235       FBXL4  OMIM:615471   
4  HP:0002188  Delayed CNS myelination          8367        H4C5  OMIM:619950   

     fuente  
0  baseline  
1  baseline  
2  baseline  
3  baseline  
4  baseline  


## Verificar ncbi_gene_id en fuentes OT (no baseline)

Las 5 primeras filas son todas baseline — ya tenían ncbi_gene_id del archivo original.
Las fuentes OT se construyeron mapeando Ensembl → gene_symbol, sin necesariamente
pasar por NCBI. Si ncbi_gene_id es NaN en esas filas, hay que decidir si importa
(depende de si database.py lo usa downstream o solo lo carga).

In [2]:
ot_full = pd.read_csv(
    Path.home() / "lucai/datos/ot_unified_genes_phenotypes.txt",
    sep="\t"
)

print("NaN en ncbi_gene_id por fuente:")
print(ot_full.groupby("fuente")["ncbi_gene_id"].apply(lambda x: x.isna().sum()))
print()
print(f"Total filas: {len(ot_full)}")
print(f"Total NaN en ncbi_gene_id: {ot_full['ncbi_gene_id'].isna().sum()}")

/tmp/ipykernel_11431/3414293425.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  ot_full = pd.read_csv(


NaN en ncbi_gene_id por fuente:
fuente
baseline          0
clingen           0
clinvar           0
gene2phenotype    0
orphanet          0
panelapp          0
uniprot_lit       0
Name: ncbi_gene_id, dtype: int64

Total filas: 1615009
Total NaN en ncbi_gene_id: 0


In [3]:
baseline = ot_full[ot_full["fuente"] == "baseline"]
genes_baseline = set(baseline["gene_symbol"].unique())
diseases_baseline = set(baseline["disease_id"].unique())

print(f"Baseline: {len(genes_baseline)} genes únicos, {len(diseases_baseline)} enfermedades únicas\n")

fuentes = ["panelapp", "clingen", "orphanet", "gene2phenotype", "uniprot_lit", "clinvar"]

for fuente in fuentes:
    df = ot_full[ot_full["fuente"] == fuente]
    genes_nuevos = set(df["gene_symbol"].unique()) - genes_baseline
    diseases_nuevas = set(df["disease_id"].unique()) - diseases_baseline
    print(f"{fuente:20} genes nuevos: {len(genes_nuevos):4d}   enfermedades nuevas: {len(diseases_nuevas):4d}")

Baseline: 5128 genes únicos, 8873 enfermedades únicas

panelapp             genes nuevos:  133   enfermedades nuevas:  161
clingen              genes nuevos:   21   enfermedades nuevas:  852
orphanet             genes nuevos:   29   enfermedades nuevas:   65
gene2phenotype       genes nuevos:   32   enfermedades nuevas:   33
uniprot_lit          genes nuevos:   90   enfermedades nuevas:   49
clinvar              genes nuevos:  438   enfermedades nuevas:  305


In [4]:
genes_nuevos_por_fuente = {}
diseases_nuevas_por_fuente = {}

for fuente in fuentes:
    df = ot_full[ot_full["fuente"] == fuente]
    genes_nuevos_por_fuente[fuente] = set(df["gene_symbol"].unique()) - genes_baseline
    diseases_nuevas_por_fuente[fuente] = set(df["disease_id"].unique()) - diseases_baseline

# Unión real (sin doble conteo)
todos_genes_nuevos = set.union(*genes_nuevos_por_fuente.values())
todas_diseases_nuevas = set.union(*diseases_nuevas_por_fuente.values())

suma_genes = sum(len(v) for v in genes_nuevos_por_fuente.values())
suma_diseases = sum(len(v) for v in diseases_nuevas_por_fuente.values())

print(f"Suma independiente (con posible doble conteo): {suma_genes} genes, {suma_diseases} enfermedades")
print(f"Unión real (sin doble conteo):                 {len(todos_genes_nuevos)} genes, {len(todas_diseases_nuevas)} enfermedades")
print(f"Solapamiento: {suma_genes - len(todos_genes_nuevos)} genes, {suma_diseases - len(todas_diseases_nuevas)} enfermedades")

Suma independiente (con posible doble conteo): 743 genes, 1465 enfermedades
Unión real (sin doble conteo):                 712 genes, 1331 enfermedades
Solapamiento: 31 genes, 134 enfermedades


## Recuperar ncbi_gene_id faltantes desde el propio dataset

Los 1048 NaN son de ClinGen. Muchos de esos genes probablemente aparecen en otras
fuentes (baseline, Orphanet, etc.) donde sí tienen ncbi_gene_id.
Estrategia: construir un diccionario gene_symbol → ncbi_gene_id desde todas las
filas que NO tienen NaN, y usarlo para rellenar los faltantes.
Si después de eso quedan genes sin cubrir, ahí buscamos en fuente externa.

In [11]:
# Diccionario gene_symbol → ncbi_gene_id desde filas que SÍ tienen valor
gene_to_ncbi = (
    ot_full[ot_full["ncbi_gene_id"].notna()]
    .drop_duplicates("gene_symbol")
    .set_index("gene_symbol")["ncbi_gene_id"]
    .to_dict()
)

# Genes con NaN
genes_nan = ot_full[ot_full["ncbi_gene_id"].isna()]["gene_symbol"].unique()
print(f"Genes únicos con ncbi_gene_id NaN: {len(genes_nan)}")

# Cuántos de esos genes están en el diccionario
recuperables = [g for g in genes_nan if g in gene_to_ncbi]
print(f"Recuperables desde el propio dataset: {len(recuperables)}")
print(f"Necesitan fuente externa: {len(genes_nan) - len(recuperables)}")

Genes únicos con ncbi_gene_id NaN: 21
Recuperables desde el propio dataset: 4
Necesitan fuente externa: 17


## Recuperar los 4 internos y ver quiénes son los 17 restantes

4 genes se recuperan cruzando con el propio dataset.
Los 17 restantes son genes que solo aparecen en ClinGen — necesitamos
ver sus nombres para elegir la fuente externa más eficiente.

In [12]:
# Aplicar recuperación interna
ot_full["ncbi_gene_id"] = ot_full.apply(
    lambda row: gene_to_ncbi.get(row["gene_symbol"], row["ncbi_gene_id"])
    if pd.isna(row["ncbi_gene_id"]) else row["ncbi_gene_id"],
    axis=1
)

# Ver quiénes quedan sin resolver
aun_faltantes = ot_full[ot_full["ncbi_gene_id"].isna()]["gene_symbol"].unique()
print(f"Genes que aún necesitan fuente externa ({len(aun_faltantes)}):")
print(sorted(aun_faltantes))

Genes que aún necesitan fuente externa (17):
['ARPP21', 'CAV2', 'DECR1', 'GEN1', 'HIBADH', 'LGALSL', 'MPST', 'NUP50', 'PLD4', 'PRODH2', 'PSD3', 'RAB40AL', 'RNU2-2', 'RNU5B-1', 'SLC12A7', 'TMEM168', 'ZFYVE27']


## Recuperar ncbi_gene_id de los 17 genes restantes via mygene.info

mygene.info es una API pública de NCBI. Acepta una lista de gene symbols
y devuelve los NCBI IDs en un solo request POST. Los dos genes de RNA
(RNU2-2, RNU5B-1) pueden no tener NCBI ID numérico — los tratamos aparte.

In [15]:
import requests

genes_a_buscar = sorted(aun_faltantes)

response = requests.post(
    "https://mygene.info/v3/query",
    data={
        "q": ",".join(genes_a_buscar),
        "scopes": "symbol",
        "fields": "entrezgene,symbol",
        "species": "human"
    }
)

print(f"Status: {response.status_code}")
resultados = response.json()

for r in resultados:
    symbol = r.get("query")
    ncbi = r.get("entrezgene")
    notfound = r.get("notfound", False)
    print(f"{symbol:15} → {ncbi if ncbi else ('NOT FOUND' if notfound else 'sin entrezgene')}")

Status: 200
ARPP21          → 10777
CAV2            → 858
DECR1           → 1666
GEN1            → 348654
HIBADH          → 11112
LGALSL          → 29094
MPST            → 4357
NUP50           → 10762
PLD4            → 122618
PRODH2          → 58510
PSD3            → 23362
RAB40AL         → 282808
RNU2-2          → 26855
RNU2-2          → sin entrezgene
RNU5B-1         → 26832
SLC12A7         → 10723
TMEM168         → 64418
ZFYVE27         → 118813


## Aplicar los IDs recuperados y guardar el archivo limpio

mygene.info resolvió los 17 genes. RNU2-2 tiene dos hits — tomamos el que
tiene entrezgene. Construimos el mapa, rellenamos los NaN restantes y
verificamos que no quede ninguno antes de sobreescribir el archivo.

In [16]:
# Construir mapa gene_symbol → ncbi_id (primer hit con entrezgene)
mapa_externo = {}
for r in resultados:
    symbol = r.get("query")
    ncbi = r.get("entrezgene")
    if ncbi and symbol not in mapa_externo:
        mapa_externo[symbol] = str(int(ncbi))

print(f"Genes mapeados: {len(mapa_externo)}")
print(mapa_externo)

# Rellenar los NaN restantes
ot_full["ncbi_gene_id"] = ot_full.apply(
    lambda row: mapa_externo.get(row["gene_symbol"], row["ncbi_gene_id"])
    if pd.isna(row["ncbi_gene_id"]) else row["ncbi_gene_id"],
    axis=1
)

print(f"\nNaN restantes: {ot_full['ncbi_gene_id'].isna().sum()}")

# Guardar
ot_full.to_csv(
    Path.home() / "lucai/datos/ot_unified_genes_phenotypes.txt",
    sep="\t",
    index=False
)
print("Archivo guardado.")

Genes mapeados: 17
{'ARPP21': '10777', 'CAV2': '858', 'DECR1': '1666', 'GEN1': '348654', 'HIBADH': '11112', 'LGALSL': '29094', 'MPST': '4357', 'NUP50': '10762', 'PLD4': '122618', 'PRODH2': '58510', 'PSD3': '23362', 'RAB40AL': '282808', 'RNU2-2': '26855', 'RNU5B-1': '26832', 'SLC12A7': '10723', 'TMEM168': '64418', 'ZFYVE27': '118813'}

NaN restantes: 0
Archivo guardado.


El dataset OT está limpio: 0 NaN, todas las columnas correctas, archivo guardado.

## Mapear simulator.py — el generador de casos sintéticos

convert_to_dataset espera un JSON con pacientes simulados.
Ese JSON lo genera el simulador, que usa database.py para leer el .txt.
Mapeamos las funciones de simulator.py para entender qué hace y cómo llamarlo.

In [17]:
%%bash
grep -n "def " ~/lucai/GraPhens/src/simulation/phenotype_simulation/simulator.py

26:    def __init__(
40:    def fit(self, data: Dict[str, np.ndarray]) -> None:
60:    def generate_patient(self, gene: str) -> List[Phenotype]:
92:    def generate_patients(self, gene_to_count: Dict[str, int], use_tqdm: bool = False) -> Dict[str, List[List[Phenotype]]]:


## Leer __init__ y fit de simulator.py

fit() es donde el simulador carga los datos de gen-fenotipo antes de generar pacientes.
Recibe Dict[str, np.ndarray] — necesitamos entender exactamente qué estructura
espera para saber cómo conectarlo con nuestro dataset OT.

In [18]:
%%bash
sed -n '26,90p' ~/lucai/GraPhens/src/simulation/phenotype_simulation/simulator.py

    def __init__(
        self, 
        distribution_strategy: DistributionStrategy, 
        phenotype_selector: PhenotypeSelector,
    ):
        """Initialize the standard phenotype simulator.
        
        Args:
            distribution_strategy: Strategy for sampling phenotype counts and distances
            phenotype_selector: Strategy for selecting phenotypes based on distances
        """
        self.distribution = distribution_strategy
        self.selector = phenotype_selector
    
    def fit(self, data: Dict[str, np.ndarray]) -> None:
        """Fit the simulator's distribution model to historical data.
        
        Args:
            data: Dictionary containing data arrays to fit the distribution
        """
        start_time = time.time()
        
        self.distribution.fit(data)
        
        total_time = time.time() - start_time
        logger.info("Fitted phenotype simulator distribution model")
        logger.debug(f"StandardPhenotypeSimulator.fit comp

## Leer factory.py de phenotype_simulation

El simulador usa dos estrategias (distribution, selector) que se instancian
en algún lugar. factory.py es donde se cablea todo junto: database → selector
→ simulator. Es el lugar más directo para entender cómo entra el .txt
al pipeline de simulación.
    

In [19]:
%%bash
grep -n "def " ~/lucai/GraPhens/src/simulation/phenotype_simulation/factory.py

26:    def create_distribution_strategy(
50:    def create_phenotype_selector(
95:    def create_simulator(
137:    def create_data_loader(
154:    def create_complete_simulator(


## Leer create_phenotype_selector y create_complete_simulator

create_phenotype_selector conecta database.py (que lee el .txt) con el selector.
create_complete_simulator cablea todo junto y es el entrypoint real del pipeline.
Estos dos métodos nos dicen exactamente qué parámetros pasar para usar nuestro
dataset OT en lugar del original.

In [20]:
%%bash
sed -n '50,180p' ~/lucai/GraPhens/src/simulation/phenotype_simulation/factory.py

    def create_phenotype_selector(
        cls,
        selector_type: str = "hpo_distance",
        gene_phenotype_facade: Optional[GenePhenotypeFacade] = None,
        hpo_provider: Optional[HPOGraphProvider] = None,
        data_dir_hpo: str = "data/ontology",
        data_dir_gene_phenotype: str = "data/gene_phenotype",
        **kwargs
    ) -> PhenotypeSelector:
        """Create a phenotype selector.
        
        Args:
            selector_type: Type of phenotype selector to create
                - "hpo_distance": HPODistancePhenotypeSelector
            gene_phenotype_facade: Optional GenePhenotypeFacade instance
            hpo_provider: Optional HPOGraphProvider instance
            data_dir_hpo: Directory for HPO data (used if hpo_provider is None)
            data_dir_gene_phenotype: Directory for gene-phenotype data (used if gene_phenotype_facade is None)
            **kwargs: Additional keyword arguments for the selector
            
        Returns:
            Phen

## Verificar si database.py necesita genes_to_phenotype.txt

data_dir_gene_phenotype tiene que contener los .txt que lee GenePhenotypeFacade.
Sabemos que carga phenotype_to_genes.txt. La línea 32 de database.py
también cargaba genes_to_phenotype.txt. Verificamos si ese archivo
es requerido o si el pipeline puede funcionar sin él.

In [21]:
%%bash
sed -n '20,50p' ~/lucai/GraPhens/src/simulation/gene_phenotype/database.py

            data_dir: Directory containing the gene-phenotype database files
        """
        self.data_dir = data_dir
        self._genes_to_phenotype = None
        self._phenotype_to_genes = None
        
    def load(self) -> None:
        """Load the gene-phenotype database files."""
        self._load_genes_to_phenotype()
        self._load_phenotype_to_genes()
        
    def _load_genes_to_phenotype(self) -> None:
        """Load the genes_to_phenotype.txt file."""
        file_path = os.path.join(self.data_dir, "genes_to_phenotype.txt")
        self._genes_to_phenotype = pd.read_csv(
            file_path, 
            sep='\t',
            dtype={
                'ncbi_gene_id': str,
                'gene_symbol': str,
                'hpo_id': str,
                'hpo_name': str,
                'frequency': str,
                'disease_id': str
            }
        )
    
    def _load_phenotype_to_genes(self) -> None:
        """Load the phenotype_to_genes.txt file.

## Verificar si frequency se usa downstream

database.py carga genes_to_phenotype.txt con una columna frequency que
nuestro dataset OT no tiene. Si el pipeline la usa, hay que decidir
qué valor poner. Si no la usa, generamos el archivo con esa columna vacía.

In [22]:
%%bash
grep -rn "frequency" ~/lucai/GraPhens/src/simulation/ | grep -v "__pycache__" | grep -v ".pyc"

/home/fran/lucai/GraPhens/src/simulation/update_goldstandard_dataset.py:197:            'frequency': '',
/home/fran/lucai/GraPhens/src/simulation/fenotipos_processed_all-MiniLM-L6-v2.json:96088:      "REVIEW:progressive high frequency SNHL"
/home/fran/lucai/GraPhens/src/simulation/fenotipos_processed_all-MiniLM-L6-v2.json:108224:      "REVIEW:and later reported to be 71% in fibroblasts, 82% in muscle, 61% in WBC, 65% in hair, and 69% in buccal.\n\nHis mother had diabetes and died at 60y after 2 stroke-like episodes. His two brothers (II-6, II-7) had diabetes. His sister (II-5) had glucose intolerance and mild high frequency HL, and the variant present at 29% blood. His healthy sister (II-1) had the variant at 31% blood",
/home/fran/lucai/GraPhens/src/simulation/fenotipos_processed_all-MiniLM-L6-v2.json:116668:      "REVIEW:First seizure at 6 months, controlled. Normal development initially, by 18 months seizure frequency increased, her seizures became drug resistant and she had clearly

grep: /home/fran/lucai/GraPhens/src/simulation/phenotype_simulation/__pycache__/distributions.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/update_goldstandard_dataset.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/__pycache__/update_deprecated_hpo.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/gene_phenotype/__pycache__/facade.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/gene_phenotype/__pycache__/demo.cpython-313.pyc: coincidencia en fichero binario
grep: /home/fran/lucai/GraPhens/src/simulation/gene_phenotype/__pycache__/database.cpython-313.pyc: coincidencia en fichero binario


 HL from 2 to 8 kHz"
/home/fran/lucai/GraPhens/src/simulation/genes_y_fenotipos.json:124999:            "Deceased at 1 month\nventriculomegaly/hydrocephalus (19 weeks gestation)\nabnormal grimacing, rhythmic extremity shaking. no spontaneous movements or eye opening\noptic nerve abnormalities elevated CK (625 U/L)\nseverely dilated lateral ventricles\nmoderate dilation of the third ventricle\nmildly prominent fourth ventricle communicating with a posterior fossa cyst, hypoplasia of the vermis, \u2018Z\u2019 shaped brainstem\ndiffuse optic nerve pallor\ncentral optic nerve excavation\nsubclinical electroencephalographic seizures arising from right occipital region\nabnormal background rhythms with poorly organized rhythms,\nhigh-voltage asynchronous theta/delta frequency pattern,\nleft or right hemispheric attenuated patterns for periods of several seconds"
/home/fran/lucai/GraPhens/src/simulation/genes_y_fenotipos.json:134597:            "high-frequency sensorineural hearing impairment

## Generar genes_to_phenotype.txt para el dataset OT

database.py requiere dos archivos: phenotype_to_genes.txt (ya tenemos) y
genes_to_phenotype.txt (no existe para OT). Este archivo mapea genes a fenotipos
con columnas: ncbi_gene_id, gene_symbol, hpo_id, hpo_name, frequency, disease_id.
frequency puede ser string vacío — el propio GraPhens usa ese fallback
(update_goldstandard_dataset.py:197). Lo generamos desde el dataset OT
reorganizando columnas y creando un directorio data_dir_gene_phenotype propio.

In [23]:
from pathlib import Path
import pandas as pd

ot_full = pd.read_csv(
    Path.home() / "lucai/datos/ot_unified_genes_phenotypes.txt",
    sep="\t"
)

# genes_to_phenotype.txt: mismas filas, columnas reordenadas + frequency vacío
genes_to_phenotype = ot_full[["ncbi_gene_id", "gene_symbol", "hpo_id", "hpo_name", "disease_id"]].copy()
genes_to_phenotype.insert(4, "frequency", "")

print("Columnas:", genes_to_phenotype.columns.tolist())
print(f"Filas: {len(genes_to_phenotype)}")
print(genes_to_phenotype.head(3))

/tmp/ipykernel_39751/1631504373.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  ot_full = pd.read_csv(


Columnas: ['ncbi_gene_id', 'gene_symbol', 'hpo_id', 'hpo_name', 'frequency', 'disease_id']
Filas: 1615009
  ncbi_gene_id gene_symbol      hpo_id                 hpo_name frequency  \
0       199857       ALG14  HP:0002188  Delayed CNS myelination             
1       199857       ALG14  HP:0002188  Delayed CNS myelination             
2         9131       AIFM1  HP:0002188  Delayed CNS myelination             

    disease_id  
0  OMIM:619031  
1  OMIM:619036  
2  OMIM:300232  


## Leer keep_leaves_of_goldstandard_dataset.py

Si Gian confirma que hay que filtrar el dataset OT antes de la simulación,
este es el script que tenemos que adaptar. Primero mapeamos sus funciones.

In [1]:
%%bash
grep -n "def " ~/lucai/GraPhens/src/simulation/keep_leaves_of_goldstandard_dataset.py

13:def setup_logging(verbose=False):
23:def get_leaf_phenotypes_for_gene(gene: str, facade: GenePhenotypeFacade, 
77:def remove_non_leaf_phenotypes(gene: str, leaf_phenotypes: Set[str], 
119:def process_all_genes(facade: GenePhenotypeFacade, 
187:def save_database(database, output_dir: Path, logger):
218:def main():


## Leer get_leaf_phenotypes_for_gene y main

get_leaf_phenotypes_for_gene define qué considera "hoja" — necesitamos entender
ese criterio para saber si aplica igual a los HPO terms de OT.
main nos dice los argumentos del script para poder llamarlo con nuestro directorio.

In [2]:
%%bash
sed -n '23,76p' ~/lucai/GraPhens/src/simulation/keep_leaves_of_goldstandard_dataset.py

def get_leaf_phenotypes_for_gene(gene: str, facade: GenePhenotypeFacade, 
                               hpo_provider: HPOGraphProvider,
                               augmentation_service: HPOAugmentationService) -> Set[str]:
    """
    Get only the most specific (leaf) phenotypes for a gene in its augmented subgraph.
    
    The process works as follows:
    1. Get all phenotypes associated with the gene from the database
    2. Use the augmentation service to build a connected subgraph by adding ancestors
       of these phenotypes (this creates the gene-specific HPO subgraph)
    3. Within this augmented subgraph, identify which of the original phenotypes 
       are leaves by checking if they have any children that are also in the 
       original set
    
    A phenotype is considered a leaf if none of its children in the augmented
    subgraph are present in the original set of phenotypes. This ensures we keep
    only the most specific phenotypes that were actually observed/r

In [3]:
%%bash
sed -n '218,280p' ~/lucai/GraPhens/src/simulation/keep_leaves_of_goldstandard_dataset.py

def main():
    parser = argparse.ArgumentParser(description="Keep only the most specific phenotypes for each gene")
    parser.add_argument(
        "--data-dir",
        default="data/",
        help="Directory containing gene-phenotype database files"
    )
    parser.add_argument(
        "--ontology-dir",
        default="data/ontology",
        help="Directory containing HPO ontology files"
    )
    parser.add_argument(
        "--output-dir",
        default="data/gene_phenotype_specific",
        help="Output directory for refined database files"
    )
    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="Run without saving changes"
    )
    parser.add_argument(
        "--verbose",
        action="store_true",
        help="Enable verbose logging"
    )
    
    args = parser.parse_args()
    logger = setup_logging(args.verbose)
    
    # Validate input directories
    data_dir = Path(args.data_dir)
    ontology_dir = Path(args.ontology_di

El filtro no es "hoja en el grafo HPO global" — es más inteligente. Un término se considera hoja si ninguno de sus descendientes HPO aparece también en el set de ese mismo gen. O sea, si el dataset OT tiene HP:0001272 (atrofia cerebelosa) y HP:0001321 (hipoplasia cerebelosa, que es descendiente), para ese gen se queda solo con HP:0001321 y elimina el más general.

## Leer create_complete_simulator completo

Necesitamos los argumentos exactos para instanciar el simulador
apuntando a nuestro directorio OT en vez del directorio original.

In [4]:
%%bash
sed -n '154,220p' ~/lucai/GraPhens/src/simulation/phenotype_simulation/factory.py

    def create_complete_simulator(
        cls,
        count_file: Optional[str] = None,
        distance_file: Optional[str] = None,
        data_dir_simulation: str = "data/simulation",
        data_dir_hpo: str = "data/ontology",
        data_dir_gene_phenotype: str = "data/gene_phenotype",
        distribution_type: str = "empirical",
        selector_type: str = "hpo_distance",
        **kwargs
    ) -> PhenotypeSimulator:
        """Create a complete simulator with data loaded from CSV files.
        
        This is a convenience method that:
        1. Creates a data loader
        2. Loads distribution data from CSV files
        3. Creates a distribution strategy and fits it with the data
        4. Creates a phenotype selector
        5. Creates and returns a simulator
        
        Args:
            count_file: Path to the phenotype count CSV file (optional)
            distance_file: Path to the phenotype distance CSV file (optional)
            data_dir_simulation: Di

## Verificar que existen los CSVs de distribución en el directorio original

In [5]:
%%bash
ls ~/lucai/GraPhens/data/simulation/

output
phenotype_counts.csv
phenotype_distances.csv
phenotype_distances_clinvar.csv
